# scTRP leave-one-out inference tutorial (rank-min ensemble)

This tutorial runs a new, external dataset through 6 leave-one-out scTRP model folds
(each fold trained holding out one of caushi/hanada/lowery/meng/oliveira/zheng) and
combines the 6 per-fold reactivity scores into one final binary call per cell using
a rank-min consensus (a cell must rank high in *every* fold to be called reactive).

**Privacy note.** No private train h5ad is used here. Each model fold ships as three
small artifacts alongside its checkpoint — `gene_panel.json`, `train_embeddings.npz`,
`val_embeddings.npz` — which contain only the model's expected gene list and derived
projector embeddings + reactivity labels, never raw/normalized expression values. These
were produced once, privately, by `prepare_fold_embeddings.py`; you should never need
to touch the original train h5ad to run this notebook.

**You need to provide:**
- `ROOT_DIR/model_folds/{study}/scTRP_simclr/` — one directory per fold, each holding
  `model_eN.pt`, `gene_panel.json`, `train_embeddings.npz`, `val_embeddings.npz`.
- Your own test dataset as an `.h5ad` (raw/log1p expression is fine; binning happens
  in this notebook).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))  # classifier/ — for infer_supcon_functions.py

from tutorial_infer_functions import (
    build_config,
    build_preprocessor,
    load_vocab_and_model_configs,
    ini_model,
    align_test_adata_to_panel,
    run_fold_inference,
    get_sample_col,
    _pool_score_matrix,
    compute_rankmin_study,
    POOL_TYPES,
)
import json

## Configuration

`ROOT_DIR` should match what you passed to `download_log1p_and_models.sh` earlier —
this notebook expects `ROOT_DIR/model_folds/{study}/scTRP_simclr/model_eN.pt` plus the
three artifact files next to it (produced by `prepare_fold_embeddings.py`).

In [ ]:
ROOT_DIR = Path("/path/to/root_dir")  # EDIT ME
TEST_H5AD_PATH = "/path/to/your_test_data.h5ad"  # EDIT ME

STUDIES = ["caushi", "hanada", "lowery", "meng", "oliveira", "zheng"]
FOLD_DIRS = {s: ROOT_DIR / "model_folds" / s / "scTRP_simclr" for s in STUDIES}

MAX_SEQ_LEN = 1200
BATCH_SIZE = 32
POOL_TYPE = "raw"  # one of tutorial_infer_functions.POOL_TYPES; "raw" = no clonotype pooling

SCORE_COL = "pro_jenks_OT_deltarho_30best_score"  # the fold-level score used for the ensemble

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = build_config(BATCH_SIZE)
vocab, model_configs = load_vocab_and_model_configs()
preprocessor = build_preprocessor(config)

## Load the test dataset once

Each fold has its own gene panel, so alignment + binning happens per fold below —
but we only read the raw test h5ad a single time here.

In [ ]:
test_adata_raw = sc.read_h5ad(TEST_H5AD_PATH)
test_adata_raw.var["gene_name"] = test_adata_raw.var.index.tolist()
print(test_adata_raw)

## Step 1: run inference for each of the 6 leave-one-out folds

For each fold: load its gene panel + precomputed train/val embeddings, align + bin the
test data to that panel, encode the test data with the fold's model, then run the full
KNN / nearest-center / distance / optimal-transport classifier battery (identical to
`final_infering()` in `infer_supcon_1004.py`) against the fold's train embeddings.

In [ ]:
fold_scores = {}
fold_full_outputs = {}

for study, fold_dir in FOLD_DIRS.items():
    print(f"\n=== fold: {study} ===")
    pt_path = sorted(Path(fold_dir).glob("model_e*.pt"))[0]

    with open(Path(fold_dir) / "gene_panel.json") as f:
        gene_panel = json.load(f)
    train_npz = np.load(Path(fold_dir) / "train_embeddings.npz", allow_pickle=True)
    val_npz = np.load(Path(fold_dir) / "val_embeddings.npz", allow_pickle=True)

    test_adata = align_test_adata_to_panel(test_adata_raw, gene_panel)
    preprocessor(test_adata, batch_key=None)

    model, epoch = ini_model(str(pt_path), vocab, model_configs, config, device=device)
    model.to(device)

    fold_df = run_fold_inference(
        model, config, MAX_SEQ_LEN, test_adata, vocab,
        train_emb=train_npz["emb"], train_labels=train_npz["reactivity"],
        val_emb=val_npz["emb"], val_labels=val_npz["reactivity"],
        device=device,
    )
    fold_full_outputs[study] = fold_df
    fold_scores[study] = fold_df[SCORE_COL]

    del model
    torch.cuda.empty_cache()

score_mat = pd.DataFrame(fold_scores)
score_mat.head()

In [ ]:
fold_scores_csv = Path(TEST_H5AD_PATH).parent / "fold_scores.csv"
score_mat.to_csv(fold_scores_csv)
print(f"wrote {fold_scores_csv}")

## Step 2: rank-min ensemble across the 6 folds

Optionally pool per-clonotype first (`POOL_TYPE`), rank each fold's column study-wide,
take each cell's minimum rank across folds as the consensus score, then binarize with a
per-sample Jenks(k=2) threshold on that consensus score.

In [ ]:
sample_s = get_sample_col(test_adata_raw.obs).reindex(score_mat.index)
sample_clone_ids = (
    test_adata_raw.obs["sample_clone_id"].reindex(score_mat.index)
    if "sample_clone_id" in test_adata_raw.obs.columns else None
)

score_mat_pooled = _pool_score_matrix(score_mat, sample_clone_ids, POOL_TYPE)
rank_min_score, rank_min_pred = compute_rankmin_study(score_mat_pooled, sample_s)

final = score_mat.copy()
final["sample"] = sample_s
if sample_clone_ids is not None:
    final["sample_clone_id"] = sample_clone_ids
final["rank_min_study"] = rank_min_score
final["pred"] = rank_min_pred

print(final["pred"].value_counts())
final.head()

In [ ]:
out_csv = Path(TEST_H5AD_PATH).parent / "rank_min_final_predictions.csv"
final.to_csv(out_csv)
print(f"wrote {out_csv}")